# 🚀 Como 1.2 MB Ajustam uma IA de 110 Milhões de Parâmetros?
### Fine-Tuning Eficiente com LoRA (BERTimbau) para Helpdesk de TI

Neste notebook, vamos desmistificar o **LoRA (Low-Rank Adaptation)** na prática:
* **Modelo Base:** `neuralmind/bert-base-portuguese-cased` (~110M parâmetros, pesando ~440 MB).
* **Adaptador LoRA:** Treinamos apenas as projeções de `query` e `value` das camadas de atenção ($r=8$).
* **Resultado:** Apenas **~295 mil parâmetros treináveis** (0.26% do modelo total) gerando um arquivo de **apenas ~1.2 MB**!
* **Caso de Uso:** Classificação de intenções de chamados de suporte técnico em português.

In [ ]:
# 1. Instalar as dependências necessárias
!pip install -q -U transformers datasets peft evaluate scikit-learn accelerate

In [ ]:
import os
import json
import torch
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)
from peft import LoraConfig, TaskType, get_peft_model, PeftModel
import evaluate

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🚀 Dispositivo em uso: {device}')

## 2. Dataset em Português: Chamados de TI / Helpdesk
Vamos criar um dataset com 6 classes principais de atendimento corporativo.

In [ ]:
LABELS = [
    'RESET_SENHA',
    'PROBLEMA_ACESSO',
    'FALHA_HARDWARE',
    'INSTALACAO_SOFTWARE',
    'LENTIDAO_REDE',
    'DUVIDA_GERAL'
]

LABEL2ID = {lbl: i for i, lbl in enumerate(LABELS)}
ID2LABEL = {i: lbl for i, lbl in enumerate(LABELS)}

raw_data = [
    {'text': 'Esqueci minha senha do Windows e não consigo logar no computador.', 'label': 'RESET_SENHA'},
    {'text': 'Minha senha expirou hoje pela manhã, como faço para renovar?', 'label': 'RESET_SENHA'},
    {'text': 'Errei a senha três vezes e minha conta corporativa foi bloqueada.', 'label': 'RESET_SENHA'},
    {'text': 'Preciso redefinir a senha do meu e-mail institucional urgente.', 'label': 'RESET_SENHA'},
    {'text': 'Não lembro qual senha cadastrei após as férias, pode resetar?', 'label': 'RESET_SENHA'},
    {'text': 'Recebi um aviso de que meu login expirou e preciso de uma nova credencial.', 'label': 'RESET_SENHA'},
    {'text': 'Poderiam enviar um link temporário para alterar minha senha?', 'label': 'RESET_SENHA'},
    {'text': 'Minha conta está bloqueada no Active Directory.', 'label': 'RESET_SENHA'},
    {'text': 'Troquei de senha ontem mas o sistema ainda não reconhece a nova.', 'label': 'RESET_SENHA'},
    {'text': 'Não consigo entrar no portal do colaborador porque esqueci as credenciais.', 'label': 'RESET_SENHA'},
    {'text': 'Solicito o desbloqueio do meu usuário no domínio corporativo.', 'label': 'RESET_SENHA'},
    {'text': 'Esqueci a chave de acesso do cofre de senhas.', 'label': 'RESET_SENHA'},
    {'text': 'Perdi meu token de autenticação em duas etapas para redefinir senha.', 'label': 'RESET_SENHA'},
    {'text': 'Onde clico para recuperar a senha da minha conta de rede?', 'label': 'RESET_SENHA'},
    {'text': 'Minha senha mestra não está funcionando no login do sistema.', 'label': 'RESET_SENHA'},
    {'text': 'Estou com erro 403 Forbidden ao tentar abrir a pasta do financeiro.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Não consigo conectar na VPN da empresa quando estou em home office.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'O ERP acusa que não tenho permissão de leitura para esse módulo.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Preciso de liberação de acesso na pasta compartilhada no servidor Z:.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Minha conta não tem autorização para visualizar os relatórios no Power BI.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Diz que meu certificado digital não tem permissão para acessar o sistema.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Fui transferido de departamento e perdi o acesso aos discos de rede.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Aparece a mensagem Acesso Negado ao abrir a ferramenta interna.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Não consigo entrar no Jira, diz que meu usuário não está no grupo correto.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Solicito permissão de administrador temporário para executar uma tarefa.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'O portal de compras não me dá acesso para aprovar orçamentos.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Estou sem acesso ao ambiente de homologação na nuvem.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Minha VPN desconecta sozinha assim que digito as credenciais.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'O time de vendas precisa de liberação imediata no CRM.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'Não tenho autorização para acessar os dados dos clientes.', 'label': 'PROBLEMA_ACESSO'},
    {'text': 'O meu monitor principal começou a piscar e agora apagou de vez.', 'label': 'FALHA_HARDWARE'},
    {'text': 'Meu notebook não liga, a luz da bateria nem acende quando coloco na tomada.', 'label': 'FALHA_HARDWARE'},
    {'text': 'O teclado do computador parou de responder, as teclas travam.', 'label': 'FALHA_HARDWARE'},
    {'text': 'O mouse sem fio parou de funcionar mesmo trocando a pilha.', 'label': 'FALHA_HARDWARE'},
    {'text': 'A ventoinha do notebook está fazendo um barulho muito forte e o aparelho esquenta.', 'label': 'FALHA_HARDWARE'},
    {'text': 'O cabo de energia da minha fonte está descascando e dando mau contato.', 'label': 'FALHA_HARDWARE'},
    {'text': 'A tela do meu laptop está com listras coloridas verticais.', 'label': 'FALHA_HARDWARE'},
    {'text': 'A impressora do segundo andar está com papel engasgado e apitando.', 'label': 'FALHA_HARDWARE'},
    {'text': 'Meu fone com microfone não é reconhecido pelo computador durante as reuniões.', 'label': 'FALHA_HARDWARE'},
    {'text': 'O dock station USB-C não está transmitindo imagem para o segundo monitor.', 'label': 'FALHA_HARDWARE'},
    {'text': 'Caiu café no teclado do meu notebook corporativo.', 'label': 'FALHA_HARDWARE'},
    {'text': 'O leitor biométrico da máquina não lê minha digital de jeito nenhum.', 'label': 'FALHA_HARDWARE'},
    {'text': 'A máquina começou a cheirar a queimado e desligou repentinamente.', 'label': 'FALHA_HARDWARE'},
    {'text': 'A webcam integrada do laptop não liga, fica tela preta no Teams.', 'label': 'FALHA_HARDWARE'},
    {'text': 'O nobreak da minha mesa está apitando continuamente.', 'label': 'FALHA_HARDWARE'},
    {'text': 'Preciso que instalem o VS Code e o Git na minha máquina de desenvolvimento.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Gostaria de solicitar uma licença do Adobe Acrobat Pro para assinar documentos.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Podem instalar o Docker Desktop para eu testar os microsserviços?', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Preciso da atualização do pacote Office 365 no meu computador.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Solicito a instalação do plugin do SAP no meu Excel.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Preciso de aprovação para instalar o DBeaver para consultar o banco de dados.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'O navegador Google Chrome precisa ser reinstalado, está corrompido.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Gostaria de pedir a instalação do Figma Desktop no meu notebook.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Precisamos instalar o antivírus corporativo na máquina que acabou de chegar.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Peço a liberação e instalação do cliente do Slack.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'O sistema pede senha de administrador local para instalar a extensão do navegador.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Preciso de um descompactador de arquivos como 7-Zip ou WinRAR.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Solicito a instalação da versão mais recente do Python e do Node.js.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Vocês poderiam instalar o software de gestão financeira na minha máquina?', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'Preciso do cliente de banco Oracle SQL Developer instalado.', 'label': 'INSTALACAO_SOFTWARE'},
    {'text': 'A internet da sala de reuniões está caindo a cada 5 minutos.', 'label': 'LENTIDAO_REDE'},
    {'text': 'O Wi-Fi da empresa está extremamente lento para carregar arquivos pesados.', 'label': 'LENTIDAO_REDE'},
    {'text': 'O sistema de faturamento está com uma lentidão anormal hoje.', 'label': 'LENTIDAO_REDE'},
    {'text': 'Não consigo carregar nenhuma página web, parece que a rede caiu por completo.', 'label': 'LENTIDAO_REDE'},
    {'text': 'A conexão via cabo na minha mesa parou de funcionar, só consigo pelo Wi-Fi.', 'label': 'LENTIDAO_REDE'},
    {'text': 'O upload de arquivos no Google Drive está demorando horas para terminar.', 'label': 'LENTIDAO_REDE'},
    {'text': 'Estamos enfrentando alta latência nas chamadas de vídeo do Teams.', 'label': 'LENTIDAO_REDE'},
    {'text': 'O ping para o servidor de produção está acima de 800ms.', 'label': 'LENTIDAO_REDE'},
    {'text': 'A navegação externa está fora do ar, apenas sites internos funcionam.', 'label': 'LENTIDAO_REDE'},
    {'text': 'Perda de pacotes constante durante a transmissão da apresentação.', 'label': 'LENTIDAO_REDE'},
    {'text': 'O roteador do terceiro andar parece que travou.', 'label': 'LENTIDAO_REDE'},
    {'text': 'A rede corporativa está instável em todo o setor de atendimento.', 'label': 'LENTIDAO_REDE'},
    {'text': 'Transferência de arquivos para o storage está a 20 KB/s, impraticável.', 'label': 'LENTIDAO_REDE'},
    {'text': 'Os downloads caem no meio do processo por timeout de rede.', 'label': 'LENTIDAO_REDE'},
    {'text': 'A intranet corporativa não responde, erro de tempo limite esgotado.', 'label': 'LENTIDAO_REDE'},
    {'text': 'Qual é o procedimento padrão para solicitar um novo equipamento?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Vocês funcionam nos finais de semana para suporte emergencial?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Onde encontro as políticas de segurança da informação da empresa?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Como faço para programar o backup automático das minhas planilhas?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Qual é o número de telefone do plantão do TI?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Como configuro a resposta automática de ausência temporária no Outlook?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Gostaria de saber qual é o prazo médio de atendimento de chamados.', 'label': 'DUVIDA_GERAL'},
    {'text': 'Posso levar o notebook da empresa para viagens internacionais a trabalho?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Onde posso descartar pilhas e equipamentos eletrônicos antigos com segurança?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Como funciona a política de uso de pendrives na empresa?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Existe algum treinamento sobre phishing disponível na intranet?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Quem é o responsável pela aprovação de novos softwares no time?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Onde abro solicitação para troca de ramal telefônico?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Como configuro minha impressora padrão no Windows 11?', 'label': 'DUVIDA_GERAL'},
    {'text': 'Qual é o link do portal de chamados oficial da empresa?', 'label': 'DUVIDA_GERAL'}
]

texts = [d['text'] for d in raw_data]
labels = [LABEL2ID[d['label']] for d in raw_data]

dataset = Dataset.from_dict({'text': texts, 'label': labels})
split = dataset.train_test_split(test_size=0.15, seed=42)
train_ds = split['train']
eval_ds = split['test']
print(f'Total de exemplos: {len(raw_data)} (Treino: {len(train_ds)}, Validação: {len(eval_ds)})')

## 3. Carregar o BERTimbau Base e Configurar o LoRA
Veja como transformamos um modelo de 110 milhões de parâmetros em apenas **~295 mil parâmetros treináveis**.

In [ ]:
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(batch['text'], truncation=True, max_length=128)

train_tokenized = train_ds.map(tokenize, batched=True)
eval_tokenized = eval_ds.map(tokenize, batched=True)

base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(LABELS),
    id2label=ID2LABEL,
    label2id=LABEL2ID
)

# Configuração LoRA
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,                  # Rank da decomposição (dimensão intermediária)
    lora_alpha=16,        # Fator de escala (scaling = alpha / r = 2.0)
    lora_dropout=0.1,
    target_modules=['query', 'value'], # Aplicado na auto-atenção
    modules_to_save=['classifier']     # Salva também a cabeça de classificação
)

model = get_peft_model(base_model, lora_config)
print('=' * 60)
model.print_trainable_parameters()
print('=' * 60)

## 4. Treinamento com Trainer do Hugging Face
Treina em poucos segundos no Colab (mesmo em CPU ou GPU gratuita).

In [ ]:
def compute_metrics(eval_pred):
    metric = evaluate.load('accuracy')
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return metric.compute(predictions=preds, references=labels)

OUTPUT_DIR = './meu-bertimbau-lora-helpdesk'

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    learning_rate=2e-4,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    save_total_limit=1,
    report_to='none'
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=eval_tokenized,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

## 5. Salvar o Adaptador e Conferir o Tamanho do Arquivo (A Prova Real!)

In [ ]:
# Salvar apenas os pesos do LoRA
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

# Verificar o tamanho do arquivo safetensors
adapter_path = os.path.join(OUTPUT_DIR, 'adapter_model.safetensors')
size_bytes = os.path.getsize(adapter_path)
size_mb = size_bytes / (1024 * 1024)

print('=' * 60)
print(f'🎉 ADAPTADOR LoRA SALVO COM SUCESSO!')
print(f'📁 Caminho: {adapter_path}')
print(f'⚖️ Tamanho exato: {size_mb:.2f} MB ({size_bytes:,} bytes)')
print(f'💡 Enquanto o BERTimbau completo pesaria ~440 MB, o LoRA pesa apenas {size_mb:.2f} MB!')
print('=' * 60)

## 6. Chatbot Interativo de Teste (Inferência em Tempo Real)
Execute esta célula para testar mensagens reais como um usuário corporativo.

In [ ]:
exemplos_teste = [
    'Meu computador travou e a tela ficou azul.',
    'Como faço para cadastrar uma nova senha de rede?',
    'A VPN não autentica quando estou em casa.',
    'Preciso da licença do Power BI para amanhã.',
    'A internet do prédio inteiro está oscilando muito.',
    'Onde vejo a política de segurança da empresa?'
]

model.eval()
print('🧪 TESTE AUTOMÁTICO DE INFERÊNCIA:\n')
for frase in exemplos_teste:
    inputs = tokenizer(frase, return_tensors='pt', truncation=True, max_length=128).to(device)
    with torch.no_grad():
        logits = model(**inputs).logits
        probs = torch.softmax(logits, dim=-1).squeeze()
        pred_idx = torch.argmax(probs).item()
        conf = probs[pred_idx].item() * 100
    
    print(f'Frase: "{frase}"')
    print(f'➡️ Intenção: {LABELS[pred_idx]} ({conf:.2f}% de confiança)\n')